# Sampling 2.000 Data Labeling dari MongoDB

Notebook ini membaca collection `comments_preprocessed`, memilih 2.000 komentar yang bersih dan layak labeling, lalu menyimpan hasilnya ke collection MongoDB baru. Sampling tidak dipaksa hanya pada satu tema; sebagian data netral tetap dipertahankan agar dataset labeling tidak bias.

## Konfigurasi

Default input: `comments_preprocessed`.

Default output: `comments_labeling_sample_2000`.

Output collection akan di-overwrite supaya hasil sampling selalu satu versi terbaru.

In [1]:
from __future__ import annotations

import json
import os
import random
import re
from collections import Counter
from datetime import datetime, timezone
from typing import Any

from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv(dotenv_path='.env', override=True)

MONGO_URI = os.getenv('MONGO_URI', '').strip()
MONGO_DATABASE = os.getenv('MONGO_DATABASE', 'analisis_sentimen').strip()
SOURCE_COLLECTION = os.getenv('LABELING_SOURCE_COLLECTION', 'comments_preprocessed').strip()
OUTPUT_COLLECTION = os.getenv('LABELING_OUTPUT_COLLECTION', 'comments_labeling_sample_2000').strip()
TARGET_SIZE = int(os.getenv('LABELING_SAMPLE_SIZE', '2000'))
RANDOM_SEED = int(os.getenv('LABELING_SAMPLE_SEED', '42'))

if not MONGO_URI:
    raise ValueError('MONGO_URI belum diisi di .env')
if TARGET_SIZE <= 0:
    raise ValueError('LABELING_SAMPLE_SIZE harus lebih dari 0')

print('database:', MONGO_DATABASE)
print('source:', SOURCE_COLLECTION)
print('output:', OUTPUT_COLLECTION)
print('target:', TARGET_SIZE)

database: analisis_sentimen
source: comments_preprocessed
output: comments_labeling_sample_2000
target: 2000


## Aturan Seleksi

Notebook ini memakai tiga bucket:

- `theme`: komentar yang kuat konteks tema publik/pemerintahan/isu TNI-DPR-RUU.
- `adjacent`: komentar masih relevan untuk sentimen sosial-politik, tetapi tidak terlalu spesifik ke tema utama.
- `neutral`: komentar yang konteksnya lebih umum atau ringan, agar dataset labeling tetap punya data netral.

Data yang sangat pendek, spam, URL, mention, duplikat teks, dan teks tanpa konteks dibuang sebelum sampling.

In [2]:
TOKEN_RE = re.compile(r'[a-z0-9_]+')

THEME_TERMS = {
    'tni', 'dpr', 'ruu', 'uu', 'militer', 'sipil', 'revisi', 'pasal', 'abri',
    'polri', 'tentara', 'prajurit', 'jenderal', 'jendral', 'menhan', 'panglima',
    'dwi', 'fungsi', 'orde', 'orba', 'reformasi', 'presiden', 'prabowo',
    'pemerintah', 'pemerintahan', 'negara', 'rakyat', 'demokrasi', 'politik',
    'jabatan', 'pejabat', 'kementerian', 'parlemen', 'mahasiswa', 'demo',
    'perampasan', 'aset', 'korupsi', 'koruptor', 'hukum', 'ham', 'pajak',
}

ADJACENT_TERMS = {
    'indonesia', 'masyarakat', 'warga', 'publik', 'pemimpin', 'pemilu', 'partai',
    'buzzer', 'deddy', 'pandji', 'panji', 'konoha', 'wakanda', 'kebijakan',
    'kekuasaan', 'adil', 'aman', 'takut', 'bahaya', 'dukung', 'tolak', 'setuju',
    'salah', 'benar', 'buruk', 'baik', 'hancur', 'rusak', 'parah', 'bagus',
    'kerja', 'uang', 'miskin', 'kaya', 'keadilan', 'suara', 'kritik', 'sistem',
}

NEUTRAL_HINTS = {
    'terima', 'kasih', 'semoga', 'amin', 'mantap', 'bagus', 'lucu', 'wkwk',
    'haha', 'nonton', 'video', 'banget', 'orang', 'saya', 'kamu', 'mereka',
}

BAD_TERMS = {
    'url_token', 'user_mention', 'togel', 'slot', 'gacor', 'casino', 'judi',
    'koislot', 'jptogel', 'mandalika', 'ambil4d', 'cuan', 'wd', 'jp', 'jepe',
}

def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall((text or '').casefold())

def has_bad_noise(text: str) -> bool:
    lowered = (text or '').casefold()
    if 'http' in lowered or 'www.' in lowered or '@' in lowered:
        return True
    tokens = set(tokenize(lowered))
    return bool(tokens & BAD_TERMS)

def quality_score(doc: dict[str, Any]) -> float:
    text_final = str(doc.get('text_final') or '').strip()
    text_original = str(doc.get('text_original') or '').strip()
    tokens = tokenize(text_final)
    unique_tokens = set(tokens)
    if not text_final or not text_original or has_bad_noise(text_final):
        return -100.0
    if len(tokens) < 4:
        return -50.0
    score = 0.0
    score += min(len(tokens), 40) * 0.9
    score += min(len(unique_tokens), 30) * 0.6
    if 6 <= len(tokens) <= 45:
        score += 12
    if len(tokens) > 80:
        score -= 18
    if len(unique_tokens) / max(len(tokens), 1) < 0.45:
        score -= 12
    if any(token.startswith('emo_') for token in tokens):
        score += 2
    return score

def context_score(doc: dict[str, Any]) -> int:
    tokens = set(tokenize(str(doc.get('text_final') or '')))
    return (len(tokens & THEME_TERMS) * 3) + len(tokens & ADJACENT_TERMS)

def bucket_for(doc: dict[str, Any]) -> str:
    tokens = set(tokenize(str(doc.get('text_final') or '')))
    theme_count = len(tokens & THEME_TERMS)
    adjacent_count = len(tokens & ADJACENT_TERMS)
    if theme_count >= 2 or (theme_count >= 1 and adjacent_count >= 2):
        return 'theme'
    if adjacent_count >= 2:
        return 'adjacent'
    return 'neutral'

def clean_output_doc(doc: dict[str, Any]) -> dict[str, Any]:
    output = {
        'comment_id': doc.get('comment_id'),
        'video_id': doc.get('video_id'),
        'text_original': str(doc.get('text_original') or '').strip(),
        'text_final': str(doc.get('text_final') or '').strip(),
    }
    if doc.get('label') is not None:
        output['label'] = doc.get('label')
    return output

def stable_sort_key(item: dict[str, Any]) -> tuple[float, int, str]:
    return (-item['_quality_score'], -item['_context_score'], str(item.get('comment_id') or ''))

## Baca Data dan Buat Kandidat

In [3]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
db = client[MONGO_DATABASE]
source = db[SOURCE_COLLECTION]
output = db[OUTPUT_COLLECTION]

raw_docs = list(source.find({}, {'_id': 0}))
print('rows source:', len(raw_docs))
if len(raw_docs) < TARGET_SIZE:
    raise ValueError(f'Data source hanya {len(raw_docs)}, kurang dari target {TARGET_SIZE}')

seen_text: set[str] = set()
seen_comment: set[str] = set()
candidates: list[dict[str, Any]] = []

for doc in raw_docs:
    comment_id = str(doc.get('comment_id') or '')
    text_key = str(doc.get('text_final') or '').strip().casefold()
    if not comment_id or not text_key:
        continue
    if comment_id in seen_comment or text_key in seen_text:
        continue
    q_score = quality_score(doc)
    if q_score < 0:
        continue
    item = dict(doc)
    item['_bucket'] = bucket_for(doc)
    item['_quality_score'] = q_score
    item['_context_score'] = context_score(doc)
    candidates.append(item)
    seen_comment.add(comment_id)
    seen_text.add(text_key)

bucket_counts = Counter(item['_bucket'] for item in candidates)
print('rows candidate:', len(candidates))
print('candidate bucket:', dict(bucket_counts))

rows source: 13446
rows candidate: 12276
candidate bucket: {'neutral': 7199, 'theme': 4678, 'adjacent': 399}


## Sampling 2.000 Data

Kuota default dibuat agar tidak terlalu bias:

- 1.100 `theme`
- 600 `adjacent`
- 300 `neutral`

Jika salah satu bucket kurang, sisa kuota diisi kandidat terbaik dari bucket lain.

In [4]:
random.seed(RANDOM_SEED)

quotas = {
    'theme': int(TARGET_SIZE * 0.55),
    'adjacent': int(TARGET_SIZE * 0.30),
}
quotas['neutral'] = TARGET_SIZE - quotas['theme'] - quotas['adjacent']

by_bucket: dict[str, list[dict[str, Any]]] = {'theme': [], 'adjacent': [], 'neutral': []}
for item in candidates:
    by_bucket[item['_bucket']].append(item)

for bucket in by_bucket:
    by_bucket[bucket].sort(key=stable_sort_key)

selected: list[dict[str, Any]] = []
selected_ids: set[str] = set()

for bucket, quota in quotas.items():
    for item in by_bucket[bucket][:quota]:
        selected.append(item)
        selected_ids.add(str(item.get('comment_id')))

if len(selected) < TARGET_SIZE:
    remaining = [item for item in candidates if str(item.get('comment_id')) not in selected_ids]
    remaining.sort(key=stable_sort_key)
    for item in remaining[: TARGET_SIZE - len(selected)]:
        selected.append(item)
        selected_ids.add(str(item.get('comment_id')))

if len(selected) > TARGET_SIZE:
    selected = selected[:TARGET_SIZE]

selected.sort(key=lambda item: (item['_bucket'], -item['_quality_score'], str(item.get('comment_id') or '')))
selected_bucket_counts = Counter(item['_bucket'] for item in selected)
print('selected rows:', len(selected))
print('selected bucket:', dict(selected_bucket_counts))
print('average quality:', round(sum(item['_quality_score'] for item in selected) / len(selected), 2))
print('average context:', round(sum(item['_context_score'] for item in selected) / len(selected), 2))

preview = [clean_output_doc(item) for item in selected[:5]]
print(json.dumps(preview, ensure_ascii=False, indent=2)[:3000])

selected rows: 2000
selected bucket: {'adjacent': 399, 'neutral': 300, 'theme': 1301}
average quality: 50.42
average context: 8.59
[
  {
    "comment_id": "Ugw8TTSyQK69I9YwivV4AaABAg",
    "video_id": "F6fgLwUeeqI",
    "text_original": "Pengalihan fokus yg dilakuin DC mirip orang yg lagi ditagih hutang\n🧔🏽 \"Bro balikin duit gw yg lu pinjem dong. Gw udah 3 kali nagih, lu bilang besok2 mulu\"\n👩🏼‍🦲\"Iya bro, ntar kalo ada duit pasti gw balikin. Lu jgn nyolot dong. Gw emang orang miskin, tp gw msh punya harga diri. Lu jadi orang gak punya empati banget ya\"\n🧔🏽 \"wtf??\"",
    "text_final": "pengalihan fokus dilakuin dc mirip orang ditagih hutang emo_other balikin uang kamu pinjem sudah kali nagih kamu bilang besok besok mulu emo_other iya ntar kalau uang balikin kamu jangan nyolot memang orang miskin punya harga diri kamu jadi orang tidak punya empati banget emo_other wtfii"
  },
  {
    "comment_id": "UgzXSOwlWy2d5pKOE2R4AaABAg.AFsXtpk-5xIAFt7Qu0ZFbl",
    "video_id": "F6fgLwUeeqI",
 

## Simpan ke MongoDB

Collection output di-overwrite. Field yang disimpan dibuat minimal agar siap dilabeli manual: `comment_id`, `video_id`, `text_original`, `text_final`, dan `label` hanya jika sudah ada di source.

In [5]:
output_docs = [clean_output_doc(item) for item in selected]
if len(output_docs) != TARGET_SIZE:
    raise RuntimeError(f'Jumlah output {len(output_docs)} tidak sama dengan target {TARGET_SIZE}')

output.drop()
output.insert_many(output_docs, ordered=False)
output.create_index('comment_id', unique=True)
output.create_index('video_id')

print('saved collection:', f'{MONGO_DATABASE}.{OUTPUT_COLLECTION}')
print('saved rows:', output.count_documents({}))
print('saved at:', datetime.now(timezone.utc).isoformat())

saved collection: analisis_sentimen.comments_labeling_sample_2000
saved rows: 2000
saved at: 2026-06-29T15:01:34.153665+00:00


## Cek Hasil

In [6]:
sample_docs = list(output.find({}, {'_id': 0}).limit(10))
print('count:', output.count_documents({}))
print('fields:', list(sample_docs[0].keys()) if sample_docs else [])
print(json.dumps(sample_docs[:3], ensure_ascii=False, indent=2)[:3000])
client.close()

count: 2000
fields: ['comment_id', 'video_id', 'text_original', 'text_final']
[
  {
    "comment_id": "Ugw8TTSyQK69I9YwivV4AaABAg",
    "video_id": "F6fgLwUeeqI",
    "text_original": "Pengalihan fokus yg dilakuin DC mirip orang yg lagi ditagih hutang\n🧔🏽 \"Bro balikin duit gw yg lu pinjem dong. Gw udah 3 kali nagih, lu bilang besok2 mulu\"\n👩🏼‍🦲\"Iya bro, ntar kalo ada duit pasti gw balikin. Lu jgn nyolot dong. Gw emang orang miskin, tp gw msh punya harga diri. Lu jadi orang gak punya empati banget ya\"\n🧔🏽 \"wtf??\"",
    "text_final": "pengalihan fokus dilakuin dc mirip orang ditagih hutang emo_other balikin uang kamu pinjem sudah kali nagih kamu bilang besok besok mulu emo_other iya ntar kalau uang balikin kamu jangan nyolot memang orang miskin punya harga diri kamu jadi orang tidak punya empati banget emo_other wtfii"
  },
  {
    "comment_id": "UgzXSOwlWy2d5pKOE2R4AaABAg.AFsXtpk-5xIAFt7Qu0ZFbl",
    "video_id": "F6fgLwUeeqI",
    "text_original": "Soalnya, mau kita setuju atau ga